# NB_DOC_03 · Cargar las descripciones en el modelo y publicar el diccionario

**Pasos 3 y 4 del framework.**

1. Toma de `descripciones.csv` solo las filas con `revisado = si`, con texto y sin `[dato a confirmar]`.
2. Guarda un **backup** de las descripciones actuales del modelo.
3. Si `DRY_RUN = False`, escribe las descripciones en el modelo con TOM (**cambio con impacto**: requiere XMLA de lectura y escritura).
4. Genera `diccionario.md` (con diagrama ER en Mermaid) y la tabla Delta `doc_diccionario` en el lakehouse para consultarla desde SQL o un reporte.

Las descripciones cargadas en el modelo se ven en Power BI como tooltip en el panel de datos y las lee Copilot.

⚠️ Si el modelo se publica desde Power BI Desktop o PBIP, la próxima publicación **pisa** estas descripciones. En ese caso llevá también los cambios al archivo fuente (ver README, sección Operación).

In [ ]:
MODELO = "<nombre-del-modelo-semantico>"
WORKSPACE = "<nombre-del-workspace>"
CARPETA_SALIDA = "/lakehouse/default/Files/doc_modelos"
DRY_RUN = True              # True = muestra qué cambiaría sin tocar el modelo
PISAR_EXISTENTES = False    # False = solo completa objetos que hoy no tienen descripción
TABLA_DELTA = "doc_diccionario"

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import json, os, re
from datetime import datetime, timezone
import pandas as pd
from sempy_labs.tom import connect_semantic_model

carpeta = os.path.join(CARPETA_SALIDA, re.sub(r"[^\w\-]+", "_", MODELO))
with open(os.path.join(carpeta, "modelo.json"), encoding="utf-8") as f:
    modelo = json.load(f)
desc = pd.read_csv(os.path.join(carpeta, "descripciones.csv"), dtype=str, keep_default_na=False)

aprobadas = desc[
    (desc["revisado"].str.lower().isin(["si", "sí"]))
    & (desc["descripcion_nueva"].str.strip() != "")
    & (~desc["descripcion_nueva"].str.contains(r"\[dato a confirmar\]"))
].copy()
print(f"{len(desc)} objetos en el CSV, {len(aprobadas)} aprobados para cargar")


def objeto_tom(tom, tipo, tabla, objeto):
    t = tom.model.Tables[tabla]
    if tipo == "tabla":
        return t
    if tipo == "columna":
        return t.Columns[objeto]
    if tipo == "medida":
        return t.Measures[objeto]
    raise ValueError(f"tipo_objeto desconocido: {tipo}")

## 1. Backup de las descripciones actuales y plan de cambios

In [ ]:
ahora = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
backup, plan = [], []

with connect_semantic_model(dataset=MODELO, workspace=WORKSPACE, readonly=True) as tom:
    for _, r in desc.iterrows():
        try:
            actual = objeto_tom(tom, r["tipo_objeto"], r["tabla"], r["objeto"]).Description or ""
        except Exception:
            print("No existe en el modelo (¿renombrado?):", r["tipo_objeto"], r["tabla"], r["objeto"])
            continue
        backup.append({**r[["tipo_objeto", "tabla", "objeto"]].to_dict(), "descripcion": actual})

    for _, r in aprobadas.iterrows():
        try:
            actual = objeto_tom(tom, r["tipo_objeto"], r["tabla"], r["objeto"]).Description or ""
        except Exception:
            continue
        nueva = r["descripcion_nueva"].strip()
        if actual == nueva or (actual and not PISAR_EXISTENTES):
            continue
        plan.append({**r[["tipo_objeto", "tabla", "objeto"]].to_dict(), "antes": actual, "despues": nueva})

ruta_backup = os.path.join(carpeta, f"backup_descripciones_{ahora}.csv")
pd.DataFrame(backup).to_csv(ruta_backup, index=False, encoding="utf-8")
print("Backup:", ruta_backup)

plan_df = pd.DataFrame(plan, columns=["tipo_objeto", "tabla", "objeto", "antes", "despues"])
print(f"Cambios a aplicar: {len(plan_df)}")
display(plan_df)

## 2. Aplicar (solo si `DRY_RUN = False`)

Al salir del bloque `with` con `readonly=False`, semantic-link-labs guarda los cambios en el servicio (`SaveChanges`). No hace falta refrescar el modelo: las descripciones son metadatos.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True: no se modificó el modelo. Revisá el plan y volvé a correr con DRY_RUN = False.")
elif plan_df.empty:
    print("No hay cambios para aplicar.")
else:
    with connect_semantic_model(dataset=MODELO, workspace=WORKSPACE, readonly=False) as tom:
        for _, r in plan_df.iterrows():
            objeto_tom(tom, r["tipo_objeto"], r["tabla"], r["objeto"]).Description = r["despues"]
    print(f"Aplicados {len(plan_df)} cambios.")

    # Validación: releer y comparar
    with connect_semantic_model(dataset=MODELO, workspace=WORKSPACE, readonly=True) as tom:
        errores = [r for _, r in plan_df.iterrows()
                   if (objeto_tom(tom, r["tipo_objeto"], r["tabla"], r["objeto"]).Description or "") != r["despues"]]
    print("Validación OK" if not errores else f"{len(errores)} objetos no quedaron como se esperaba")

## 3. Publicar: `diccionario.md` y tabla Delta `doc_diccionario`

La descripción publicada es la misma que queda (o quedaría, en `DRY_RUN`) en el modelo: la aprobada si corresponde cargarla, si no la que ya tenía el modelo, si no `[dato a confirmar]`.

In [ ]:
def descripcion_final(tipo, tabla, objeto, actual):
    m = aprobadas[(aprobadas["tipo_objeto"] == tipo) & (aprobadas["tabla"] == tabla) & (aprobadas["objeto"] == objeto)]
    # Misma regla que el plan: sin PISAR_EXISTENTES, la descripción que ya tenía el modelo se respeta
    if not m.empty and (not actual or PISAR_EXISTENTES):
        return m.iloc[0]["descripcion_nueva"].strip()
    return actual or "[dato a confirmar]"


def sensibilidad(tabla, objeto):
    m = desc[(desc["tipo_objeto"] == "columna") & (desc["tabla"] == tabla) & (desc["objeto"] == objeto)]
    return (m.iloc[0]["sensibilidad"] if not m.empty else "") or "[dato a confirmar]"


def celda(v):
    """Escapa texto para una celda de tabla Markdown."""
    return str(v).replace("|", "\\|").replace("\n", " ") if v not in (None, "") else "-"


def entidad(nombre):
    return re.sub(r"\W+", "_", nombre).strip("_") or "tabla"


relaciones_por_col = {}
for r in modelo["relaciones"]:
    relaciones_por_col.setdefault((r["desde_tabla"], r["desde_columna"]), []).append(f'→ {r["hacia_tabla"]}[{r["hacia_columna"]}]')
    relaciones_por_col.setdefault((r["hacia_tabla"], r["hacia_columna"]), []).append(f'← {r["desde_tabla"]}[{r["desde_columna"]}]')

CARD = {("Many", "One"): "}o--||", ("One", "Many"): "||--o{", ("One", "One"): "||--||", ("Many", "Many"): "}o--o{"}

md = [f"# Diccionario de datos: {MODELO}", "",
      f"Origen: modelo semántico de Microsoft Fabric · Workspace: {WORKSPACE} · Generado: {datetime.now(timezone.utc):%Y-%m-%d}", "",
      "## Diagrama de relaciones", "", "```mermaid", "erDiagram"]
for r in modelo["relaciones"]:
    rel = CARD.get((r["desde_cardinalidad"], r["hacia_cardinalidad"]), "}o--||")
    etiqueta = r["desde_columna"] + ("" if r["activa"] else " (inactiva)")
    md.append(f'    {entidad(r["desde_tabla"])} {rel} {entidad(r["hacia_tabla"])} : "{etiqueta}"')
md += ["```", "", "## Tablas", "", "| Tabla | Filas | Modo | Descripción |", "| --- | --- | --- | --- |"]
for t in modelo["tablas"]:
    md.append(f'| {celda(t["nombre"])} | {celda(t.get("filas"))} | {celda(t.get("modo"))} | '
              f'{celda(descripcion_final("tabla", t["nombre"], t["nombre"], t["descripcion_actual"]))} |')

filas_delta = []
for t in modelo["tablas"]:
    md += ["", f'## {t["nombre"]}', "", descripcion_final("tabla", t["nombre"], t["nombre"], t["descripcion_actual"]), "",
           "| Columna | Tipo | Descripción | Ejemplo | Nulos | Clave | Relación | Sensibilidad |",
           "| --- | --- | --- | --- | --- | --- | --- | --- |"]
    for c in t["columnas"]:
        d = descripcion_final("columna", t["nombre"], c["nombre"], c["descripcion_actual"])
        s = sensibilidad(t["nombre"], c["nombre"])
        ejemplo = "-" if s == "Sensible" or c.get("sensible_sugerido") else ", ".join(c.get("ejemplos", [])[:2])
        nulos = f'{c["nulos"]} ({c["pct_nulos"]}%)' if c.get("nulos") is not None and c.get("pct_nulos") is not None else c.get("nulos")
        rels = "; ".join(relaciones_por_col.get((t["nombre"], c["nombre"]), []))
        md.append(f'| {celda(c["nombre"])} | {celda(c["tipo_dato"])} | {celda(d)} | {celda(ejemplo)} | '
                  f'{celda(nulos)} | {"Sí" if c["es_clave"] else "-"} | {celda(rels)} | {celda(s)} |')
        filas_delta.append({"modelo": MODELO, "workspace": WORKSPACE, "tipo_objeto": "columna", "tabla": t["nombre"],
                            "objeto": c["nombre"], "tipo_dato": c["tipo_dato"], "descripcion": d, "sensibilidad": s,
                            "nulos": c.get("nulos"), "cardinalidad": c.get("cardinalidad"), "expresion": c.get("expresion")})
    filas_delta.append({"modelo": MODELO, "workspace": WORKSPACE, "tipo_objeto": "tabla", "tabla": t["nombre"],
                        "objeto": t["nombre"], "tipo_dato": None,
                        "descripcion": descripcion_final("tabla", t["nombre"], t["nombre"], t["descripcion_actual"]),
                        "sensibilidad": None, "nulos": None, "cardinalidad": t.get("filas"), "expresion": None})

if modelo["medidas"]:
    md += ["", "## Medidas", "", "| Medida | Tabla | Descripción | Formato | Expresión DAX |", "| --- | --- | --- | --- | --- |"]
    for m in modelo["medidas"]:
        d = descripcion_final("medida", m["tabla"], m["nombre"], m["descripcion_actual"])
        md.append(f'| {celda(m["nombre"])} | {celda(m["tabla"])} | {celda(d)} | {celda(m["formato"])} | `{celda(m["expresion"])}` |')
        filas_delta.append({"modelo": MODELO, "workspace": WORKSPACE, "tipo_objeto": "medida", "tabla": m["tabla"],
                            "objeto": m["nombre"], "tipo_dato": None, "descripcion": d, "sensibilidad": None,
                            "nulos": None, "cardinalidad": None, "expresion": m["expresion"]})

ruta_md = os.path.join(carpeta, "diccionario.md")
with open(ruta_md, "w", encoding="utf-8") as f:
    f.write("\n".join(md) + "\n")
print("Diccionario:", ruta_md)

In [ ]:
# Tabla Delta en el lakehouse por defecto: reemplaza solo las filas de este modelo
from pyspark.sql import functions as F

pdf = pd.DataFrame(filas_delta).astype(str).replace({"None": None, "nan": None})
df = spark.createDataFrame(pdf).withColumn("publicado_utc", F.current_timestamp())
if spark.catalog.tableExists(TABLA_DELTA):
    spark.sql(f"DELETE FROM {TABLA_DELTA} WHERE modelo = '{MODELO.replace(chr(39), chr(39) * 2)}'")
    df.write.mode("append").saveAsTable(TABLA_DELTA)
else:
    df.write.saveAsTable(TABLA_DELTA)
print(f"{df.count()} filas publicadas en {TABLA_DELTA}")

## Volver atrás

Para restaurar las descripciones anteriores, cargá el backup en `desc_backup` y aplicalo igual que arriba:

```python
desc_backup = pd.read_csv(os.path.join(carpeta, "backup_descripciones_<AAAAMMDDTHHMMSSZ>.csv"), dtype=str, keep_default_na=False)
with connect_semantic_model(dataset=MODELO, workspace=WORKSPACE, readonly=False) as tom:
    for _, r in desc_backup.iterrows():
        objeto_tom(tom, r["tipo_objeto"], r["tabla"], r["objeto"]).Description = r["descripcion"]
```